# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


# Exercise:

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
# Import required libraries

import pandas as pd
import re
import string
import nltk

# NLTK tools for stopwords, lemmatization, and stemming
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [12]:
# Download required NLTK resources

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

Load dataset:

In [13]:
data = pd.read_csv("/content/drive/MyDrive/AI-Sem6/week8/trum_tweet_sentiment_analysis.csv",encoding="ISO-8859-1")

# Show first 5 rows
data.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


Check columns and missing values:

In [14]:
# Check column names and dataset size

print("Columns:", data.columns.tolist())
print("Dataset shape:", data.shape)

# Check missing values
print("\nMissing values:")
print(data.isnull().sum())

Columns: ['text', 'Sentiment']
Dataset shape: (1850123, 2)

Missing values:
text         0
Sentiment    0
dtype: int64


Create stopwords, lemmatizer, and stemmer

In [15]:
# Stopwords are common words like "the", "is", "and"
stop_words = set(stopwords.words("english"))

# Lemmatizer changes words to their base form
wordnet = WordNetLemmatizer()

# Stemmer cuts words to their root form
porter = PorterStemmer()

## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

In [16]:
# Convert text to lowercase
def lower_order(text):
    return str(text).lower()


# Remove URLs from text
def remove_urls(text):
    return re.sub(r"http\S+|www\S+|https\S+", "", str(text))


# Remove emojis from text
def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols and pictures
        u"\U0001F680-\U0001F6FF"  # transport symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002700-\U000027BF"  # other symbols
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub("", str(text))


# Remove mentions, hashtag symbol, numbers, punctuation, and extra spaces
def removeunwanted_characters(text):
    text = re.sub(r"@\w+", "", str(text))      # remove @mentions
    text = re.sub(r"#", "", text)              # remove hashtag symbol only
    text = re.sub(r"\d+", "", text)            # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()   # remove extra spaces
    return text

# Build a Text Cleaning Pipeline

In [19]:
def text_cleaning_pipeline(text, rule="lemmatize"):
    """
    This function cleans raw text data.
    Steps:
    lowercase -> remove URLs -> remove emojis -> remove unwanted characters
    -> tokenize -> remove stopwords -> lemmatize/stem -> return cleaned text
    """
    # Convert text to lowercase
    text = lower_order(text)

    # Remove URLs
    text = remove_urls(text)

    # Remove emojis
    text = remove_emoji(text)

    # Remove unwanted characters
    text = removeunwanted_characters(text)

    # Tokenize text into words
    tokens = text.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Apply lemmatization or stemming
    if rule == "lemmatize":
        tokens = [wordnet.lemmatize(word, pos="v") for word in tokens]

    elif rule == "stem":
        tokens = [porter.stem(word) for word in tokens]

    else:
        raise ValueError("Rule must be either 'lemmatize' or 'stem'")

    return " ".join(tokens)

In [20]:
# Test the cleaning function

sample = "Hello @gabe_flomo 👋🏾, I still want us to hit that new sushi spot??? LMK when you're free!!! #sushiBros #🍱"

print("Original text:")
print(sample)

print("\nCleaned text:")
print(text_cleaning_pipeline(sample))

Original text:
Hello @gabe_flomo 👋🏾, I still want us to hit that new sushi spot??? LMK when you're free!!! #sushiBros #🍱

Cleaned text:
hello still want us hit new sushi spot lmk youre free sushibros


Apply cleaning functions:

In [21]:
# Apply the text cleaning pipeline to the tweet text column

data["cleaned_text"] = data["text"].apply(lambda text: text_cleaning_pipeline(text))

# Show original text, cleaned text, and sentiment
data[["text", "cleaned_text", "Sentiment"]].head()

,text,cleaned_text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,rt trump drain swamp taxpayer dollars trip adv...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,icymi hackers rig fm radio station play antitr...,0
2,Trump protests: LGBTQ rally in New York https:...,trump protest lgbtq rally new york bbcworld via,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",hi im piers morgan david beckham awful donald ...,0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,rt tech firm sue buzzfeed publish unverified t...,0


Define features and labels:

In [22]:
X = data["cleaned_text"] # X = cleaned tweet text
y = data["Sentiment"] # y = sentiment label

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1850123,)
y shape: (1850123,)


Train-test split:

In [23]:
# Split the dataset into training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,                  # Input feature: cleaned tweet text
    y,                  # Targetlabel: sentiment
    test_size=0.2,     # 20% testing, 80% training
    random_state=42,    # Makes the split same every time we run the code
    stratify=y          # Keeps the same sentiment class ratio in train and test sets
)

print("Training data size:", X_train.shape)
print("Testing data size:", X_test.shape)

Training data size: (1480098,)
Testing data size: (370025,)


TF- IDF vectorization:

In [24]:
# Convert text into numerical values using TF-IDF

tfidf = TfidfVectorizer(max_features=5000)

# Learn vocabulary from training text and convert it into numbers
X_train_tfidf = tfidf.fit_transform(X_train)

# Convert test text into numbers using same vocabulary
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (1480098, 5000)
Testing TF-IDF shape: (370025, 5000)


Train logistic regression model:

It learns patterns between tweet words and sentiment labels.

In [25]:
# Create Logistic Regression model
model = LogisticRegression(max_iter=1000)

# Train the model
model.fit(X_train_tfidf, y_train)

print("Model training completed.")

Model training completed.


Make prediction:

In [26]:
# Predict sentiment for test data

y_pred = model.predict(X_test_tfidf)

print("Prediction completed.")

Prediction completed.


Evaluate model:

In [27]:
# Check model performance

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9225592865346937

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.95      0.94    248842
           1       0.90      0.86      0.88    121183

    accuracy                           0.92    370025
   macro avg       0.92      0.91      0.91    370025
weighted avg       0.92      0.92      0.92    370025


Confusion Matrix:
[[237306  11536]
 [ 17119 104064]]


Test model:

In [28]:
# Test the model using a new custom tweet

new_tweet = ["Trump is doing a great job and people love his speech!"]

# Clean the new tweet
new_tweet_cleaned = [text_cleaning_pipeline(tweet) for tweet in new_tweet]

# Convert cleaned tweet into TF-IDF numbers
new_tweet_tfidf = tfidf.transform(new_tweet_cleaned)

# Predict sentiment
prediction = model.predict(new_tweet_tfidf)

print("Original tweet:", new_tweet[0])
print("Cleaned tweet:", new_tweet_cleaned[0])
print("Predicted sentiment:", prediction[0])

Original tweet: Trump is doing a great job and people love his speech!
Cleaned tweet: trump great job people love speech
Predicted sentiment: 1


Save cleaned dataset:

In [29]:
# Save the dataset with cleaned_text column

data.to_csv("/content/drive/MyDrive/AI-Sem6/week8/cleaned_trump_sentiment_dataset.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
